In [0]:
# Step 1: Install LangChain core + community packages
%pip install langchain==0.2.16
%pip install langchain-community==0.2.16
%pip install langchain-core==0.2.38

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 112.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 607.6/607.6 kB 33.1 MB/s eta 0:00:00
  Attempting uninstall: tenacity
    Found existing installation: tenacity 9.0.0
    Not uninstalling tenacity at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-6cce3145-6996-4f9f-a4d4-d1aef325b9c7
    Can't uninstall 'tenacity'. No files were found to uninstall.
  Attempting uninstall: numpy
    Found existing installation: numpy 2.1.3
    Not uninstalling numpy at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-6cce3145-6996-4f9f-a4d4-d1aef325b9c7
    Can't uninstall 'nu

In [0]:
# Step 2: Install Databricks-specific integrations (for embeddings & vector search)
%pip install "langchain-databricks>=0.1.0"
%pip install databricks-vectorsearch
# Step 3: Install FAISS (vector database - CPU version works perfectly in Databricks)
%pip install faiss-cpu
# Step 4: Install Azure OpenAI integration for LangChain
%pip install langchain-openai
# Step 5: Optional but recommended - for better pandas handling and progress bars
%pip install tqdm pandas

INFO: pip is looking at multiple versions of mlflow to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of mlflow to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 113.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 120.2 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Not uninstalling protobuf at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-6cce3145-6996-4f9f-a4d4-d1aef325b9c7
    Can't uninstall 'protobuf'. No files we

In [0]:
%pip install langchain==0.2.16 langchain-community==0.2.16 langchain-openai==0.1.22 langchain-databricks faiss-cpu --force-reinstall -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.67.0 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.8 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
# %pip install langchain langchain-openai langchain-community faiss-cpu
# Restart Python kernel after install if needed (Runtime -> Restart Python)


import pandas as pd
from langchain.docstore.document import Document
from langchain_community.vectorstores import FAISS
from langchain_databricks.embeddings import DatabricksEmbeddings
# Fixed import: Use AzureOpenAIEmbeddings for embeddings (not Chat-specific) + AzureChatOpenAI for LLM
from langchain_openai import AzureOpenAIEmbeddings, AzureChatOpenAI
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.prompts import PromptTemplate

# Verify the fix
print("Imports successful! AzureOpenAIEmbeddings available:", hasattr(AzureOpenAIEmbeddings, '__call__'))



Imports successful! AzureOpenAIEmbeddings available: True


In [0]:
# =========================
# 1. Load the Gold Table into Pandas DataFrame
# =========================
df_pandas = spark.table("hrcatalog.lakehouse.gold_hr_all_employees").toPandas()


In [0]:
# =========================
# 2. Create LangChain Documents (one document per employee row)
# =========================
documents = []
for idx, row in df_pandas.iterrows():
    # Build a clean, readable string for each employee
    content_lines = []
    for col in df_pandas.columns:
        val = row[col]
        if pd.notna(val) and val != "" and val != "null":
            # Make column names human-readable
            clean_col = col.replace("_", " ").title()
            content_lines.append(f"{clean_col}: {val}")
    
    content = "\n".join(content_lines)
    
    documents.append(
        Document(
            page_content=content,
            metadata={"EmpID": row.get("EmpID", "Unknown"), "Department": row.get("Department", "Unknown")}
        )
    )

print(f"Created {len(documents)} employee documents created for RAG.")


Created 5809 employee documents created for RAG.


In [0]:
# =========================
# 3. Create Vector Store (using Databricks Foundation Model Embeddings - no extra cost / API key needed)
# =========================
# Recommended endpoint: "databricks-bge-large-en" or "databricks-gte-large-en"
# Both are excellent and free to use in your workspace
embeddings = DatabricksEmbeddings(endpoint="databricks-bge-large-en")

# Build the FAISS index (in-memory, fast for ~1.5k rows)
vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever = vectorstore.as_retriever(search_kwargs={"k": 50})  # 50 to help with aggregate questions



# # =========================
# # 3. Create Vector Store — BUT DO NOT LIMIT RETRIEVAL!
# # =========================
# embeddings = DatabricksEmbeddings(endpoint="databricks-bge-large-en")

# vectorstore = FAISS.from_documents(documents, embeddings)

# # THIS IS THE KEY CHANGE: Retrieve ALL documents (5809) instead of just top-15
# retriever = vectorstore.as_retriever(
#     search_type="similarity",
#     search_kwargs={"k": len(documents)}  # k = total number of docs
# )

# print(f"Retriever configured to return ALL {len(documents)} employee records for every question.")


In [0]:
# =========================
# 4. Setup Azure OpenAI GPT-4o LLM
# =========================
llm = AzureChatOpenAI(
    azure_endpoint="https://parka-mi5evr98-norwayeast.cognitiveservices.azure.com/",  # Base resource URL (without /openai/deployments/...)
    api_key="5wSTBg2cEKY8uWzyTr1JwuF9dtmM39iwK9YvO7IL9kmFo350SlxfJQQJ99BKAChHRaEXJ3w3AAAAACOGJCur",
    azure_deployment="gpt-4o_parkaru37",
    api_version="2024-05-01-preview",
    temperature=0.2,
    max_tokens=1000
)



In [0]:

# =========================
# 5. Custom Prompt for Better HR Answers
# =========================
prompt_template = """You are an expert HR Analytics assistant with access to the full employee dataset.
Use ONLY the context provided below to answer the question. 

Context (employee records):
{context}

Question: {question}

Instructions:
- Answer factually using the provided records.
- If the question is about averages, counts, attrition rate, etc., calculate/estimate from the retrieved records (all employees in the same department have identical department KPIs).
- For specific employees, mention their EmpID.
- If you cannot answer with certainty, say "Not enough information in the retrieved records".
- Keep answers concise and professional.

Answer:"""

PROMPT = PromptTemplate(template=prompt_template, input_variables=["context", "question"])



In [0]:
# ========================================
# 6. FINAL & FULLY WORKING RAG CHAIN (2025-compatible)
# ========================================

from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain.memory import ConversationBufferMemory
from langchain_core.runnables import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# --- Prompt ---
system_prompt = """You are an expert HR Analytics assistant.
Use ONLY the following employee records to answer the question accurately and professionally.

Context:
{context}

Instructions:
- Answer factually using only the retrieved employee records.
- For counts, averages, attrition rates, etc., calculate from the retrieved records.
- Always mention EmpID when referring to a specific employee.
- If you cannot answer with certainty, say "Not enough information".
- Keep answers concise and clear.

Answer:"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

# --- Create the two chains ---
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# --- Add conversation memory (so follow-ups work) ---
memory = ChatMessageHistory()

conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    lambda session_id: memory,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer"
)

print("RAG Chain with conversation memory is READY!")

# # =========================
# # 6. FINAL RAG CHAIN (Uses ALL 5809 records!)
# # =========================
# from langchain.chains import create_retrieval_chain
# from langchain.chains.combine_documents import create_stuff_documents_chain
# from langchain_core.prompts import ChatPromptTemplate
# from langchain_core.runnables import RunnableWithMessageHistory
# from langchain_community.chat_message_histories import ChatMessageHistory

# # Improved prompt — tells LLM we have the FULL dataset
# system_prompt = """You are an expert HR Analytics assistant with access to the COMPLETE employee dataset (5,809 records).

# You have been given ALL relevant employee records in the context below — no filtering has been applied.

# Use this full context to answer accurately, especially for:
# - Counts, averages, percentages, attrition rates
# - Department-level statistics
# - Lists of employees meeting criteria
# - Highest/lowest values

# Context (FULL dataset or large relevant subset):
# {context}

# Answer factually and professionally. Always base calculations on the provided data."""

# prompt = ChatPromptTemplate.from_messages([
#     ("system", system_prompt),
#     ("human", "{input}")
# ])

# question_answer_chain = create_stuff_documents_chain(llm, prompt)
# rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# # Add memory
# memory = ChatMessageHistory()

# conversational_rag_chain = RunnableWithMessageHistory(
#     rag_chain,
#     lambda session_id: memory,
#     input_messages_key="input",
#     history_messages_key="chat_history",
#     output_messages_key="answer"
# )

# print("RAG Chain READY — using ALL 5,809 employee records!")

RAG Chain with conversation memory is READY!


In [0]:
# ========================================
# 7. INTERACTIVE CHATBOT (Run this and start asking!)
# ========================================

print("\n" + "="*70)
print("HR Analytics RAG Chatbot is LIVE!")
print("Examples:")
print("   • Who has the highest salary?")
print("   • What is the attrition rate in Sales?")
print("   • List employees who work overtime")
print("   • Average age in Research & Development?")
print("Type 'exit' or 'quit' to stop.")
print("="*70 + "\n")

while True:
    question = input("You: ").strip()
    
    if question.lower() in ["exit", "quit", "bye"]:
        print("Goodbye!")
        break
    if not question:
        continue

    try:
        result = conversational_rag_chain.invoke(
            {"input": question},
            config={"configurable": {"session_id": "hr_chat"}}
        )
        
        answer = result["answer"]
        print(f"\nBot: {answer}\n")

        # Show which employees were used
        docs = result.get("context", [])
        emp_ids = [doc.metadata.get("EmpID", "N/A") for doc in docs]
        unique_ids = list(set(emp_ids))
        if len(unique_ids) <= 8:
            print(f"   Based on EmpID: {', '.join(unique_ids)}\n")
        else:
            print(f"   Based on {len(unique_ids)} employees\n")

    except Exception as e:
        print(f"Error: {e}")


# # =========================
# # 7. CHATBOT — Now shows real data size
# # =========================
# print("\n" + "="*80)
# print("HR Analytics RAG Chatbot — FULL DATASET MODE (5,809 employees)")
# print("Now using ALL employee records for every question!")
# print("="*80 + "\n")

# while True:
#     question = input("You: ").strip()
#     if question.lower() in ["exit", "quit", "bye"]:
#         print("Goodbye!")
#         break
#     if not question:
#         continue

#     try:
#         result = conversational_rag_chain.invoke(
#             {"input": question},
#             config={"configurable": {"session_id": "hr_full"}}
#         )
        
#         answer = result["answer"]
#         print(f"\nBot: {answer}\n")

#         # Show real count
#         docs_used = len(result.get("context", []))
#         print(f"   Based on {docs_used:,} employee records\n")

#     except Exception as e:
#         print(f"Error: {e}")



HR Analytics RAG Chatbot is LIVE!
Examples:
   • Who has the highest salary?
   • What is the attrition rate in Sales?
   • List employees who work overtime
   • Average age in Research & Development?
Type 'exit' or 'quit' to stop.



You:  give me confusion matrix


Bot: Not enough information. The provided employee records do not include predicted vs. actual attrition data necessary to construct a confusion matrix.

   Based on 50 employees



You:  give me average salary


Bot: The average monthly income across the provided employee records is **₹9,928.81**.

   Based on 50 employees



...


Bot: Based on the retrieved employee records, there are **3 departments**:  
1. **Research & Development**  
2. **Human Resources**  
3. **Healthcare Representative**

   Based on 50 employees



You:  List all departments and their headcount.


Bot: Based on the provided employee records, the departments and their headcounts are as follows:

1. **Research & Development**: 48 employees  
2. **Human Resources**: 13 employees  

Total headcount: 61 employees.

   Based on 50 employees



You:  exit

Goodbye!
